In [0]:
%run /Users/sridhar.sathyanarayanan@celebaltech.com/DemoRepo/MyFolder/Copy-Datasets

In [0]:
files = dbutils.fs.ls(f"{bookstore.dataset_path}/kafka-raw")
display(files)

In [0]:
df_raw = spark.read.json(f"{bookstore.dataset_path}/kafka-raw")
display(df_raw)

In [0]:
from pyspark.sql import functions as F

def process_bronze():
  
    schema = "key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG"

    query = (spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format", "json")
                        .schema(schema)
                        .load(f"{bookstore.dataset_path}/kafka-raw")
                        .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))  
                        .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
                  .writeStream
                      .option("checkpointLocation", f"{bookstore.checkpoint_path}/bronze")
                      .option("mergeSchema", True)
                      .partitionBy("topic", "year_month")
                      .trigger(availableNow=True)
                      .table("bronze"))
    
    query.awaitTermination()

In [0]:
process_bronze()

In [0]:
batch_df = spark.table("bronze")
display(batch_df)

In [0]:
%sql
SELECT * FROM bronze

In [0]:
%sql
SELECT DISTINCT topic from bronze

In [0]:
bookstore.load_new_data()

In [0]:
%sql
SELECT COUNT(*) FROM bronze

In [0]:
%sql
select v.* from(
select from_json(cast(value as String),"order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
from bronze
where topic = "orders")

In [0]:
(   spark.readStream
            .table("bronze")
            .createOrReplaceTempView("bronze_tmp"))

In [0]:
from pyspark.sql import functions as F

json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"

query = (spark.readStream.table('bronze')
        .filter("topic = 'orders'")
        .select(F.from_json(F.col("value").cast("string"),json_schema).alias("v"))
        .select("v.*")
    .writeStream
    .option("checkpointLocation",f"{bookstore.dataset_path}/orders_silver")
    .trigger(availableNow = True)
    .table("orders_silver")
    )

query.awaitTermination()
        
        

In [0]:
%sql
select * from orders_silver